# 과거 v17 실험 회귀 워크벤치
이 노트북은 초기 pressure EB 모듈의 실행 계약을 회귀 테스트하기 위해 보존합니다. 최종 v345 학습·재현 진입점이 아닙니다.

최종 재현은 제출 폴더 루트의 `00_RUN_REPRODUCTION.ipynb`와 `03_REPRODUCE.md`를 사용하세요. 이 노트북은 기본값 `DRY_RUN=True`에서 명령만 출력합니다.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'tests').is_dir():
            return candidate
    raise FileNotFoundError('code/src와 code/tests가 있는 프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f'project: {PROJECT_ROOT}')
print(f'python : {sys.executable}')
print(f'version: {sys.version.split()[0]}')

## 실행 스위치

- `DRY_RUN`: `True`이면 명령만 출력합니다.
- `RUN_HEAVY`: 과거 screen·학습·강건 평가 명령을 허용합니다.
- `RUN_PACKAGING`: 과거 v17 구조의 패키징 검증을 허용합니다.

최종 v345 재현에는 이 스위치를 사용하지 않습니다.

In [ ]:
DRY_RUN = True
RUN_TESTS = True
RUN_HEAVY = False
RUN_PACKAGING = False
RUN_ID = 'v19_regression_20260816_01'

V17_PUBLIC = 1093.3213473808
HISTORICAL_SCOPE = 'v17 regression only'
V17_SELECTED = 'multi_pitcher_batter_hand_pressure_d1_a3200_w1.5'
V17_SCREEN_DIR = 'v16_multiseason_20260815_02'
V14_SHA256 = 'AC8E135FBA8DBF41E331A8F4E2AAA658F0AF6675DD5F3B6C89EF51E9BC02977E'

print(f'historical reference score: {V17_PUBLIC:.10f}')

In [ ]:
def command_text(command):
    return subprocess.list2cmdline([str(part) for part in command])


def run_command(command, *, enabled, label):
    command = [str(part) for part in command]
    print(f'[{label}] {command_text(command)}')
    if DRY_RUN:
        print('  DRY_RUN: 실행하지 않음')
        return None
    if not enabled:
        print('  실행 스위치가 꺼져 있음')
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, check=True)


def run_module(module, *args, enabled, label):
    return run_command(
        [sys.executable, '-m', module, *args], enabled=enabled, label=label
    )

## 과거 입력 경로 점검

초기 계보 회귀 실행에 사용됐던 경로의 존재 여부만 표시합니다. 파일 내용을 읽거나 출력하지 않습니다.

In [ ]:
required_paths = {
    'train data': Path('data/train.csv'),
    'test data': Path('data/test.csv'),
    'v14 parent': Path('submit_v14.zip'),
    'v17 champion': Path('submit_v17.zip'),
    'v17 screen cache': Path('artifacts') / V17_SCREEN_DIR,
    'v17 residual artifact': Path('artifacts/v17_residual_final_20260815'),
}
for name, path in required_paths.items():
    state = 'READY' if path.exists() else 'MISSING'
    print(f'{state:7}  {name:24}  {path}')

## 보존 범위

과거 제출 이력 파일은 최종 재현 패키지에 포함하지 않습니다. 아래 셀은 이 노트북의 회귀 검증 목적을 명시합니다.

In [ ]:
print('과거 v17 모듈 호출 구조만 보존하며 제출 이력은 읽지 않습니다.')

## 데이터 비의존 회귀 테스트

이 노트북 자체의 형식·문법·DRY_RUN 계약을 검사합니다.

In [ ]:
run_module(
    'pytest', '-q', 'tests/test_experiment_notebook.py',
    enabled=RUN_TESTS, label='notebook contract tests',
)

## 후보 축 screen

출력은 고유한 `artifacts/` 하위 폴더에 기록합니다. 최고 한 점이 아니라 최근 전이 최소 gain과 의존성 강건성을 함께 판정합니다.

In [ ]:
run_module(
    'src.v16_residual_calibration_screen',
    '--output-dir', f'artifacts/{RUN_ID}_residual_calibration',
    enabled=RUN_HEAVY, label='residual calibration screen',
)
run_module(
    'src.archive.v16_multiseason_screen',
    '--output-dir', f'artifacts/{RUN_ID}_multiseason',
    enabled=RUN_HEAVY, label='multi-season EB screen',
)

## v17 선택 recipe 회귀 평가

고정된 초기 recipe가 기존 강건 평가 모듈로 연결되는지 확인합니다.

In [ ]:
run_module(
    'src.evaluate_v16_robust',
    '--selected', V17_SELECTED,
    '--artifact-dir', V17_SCREEN_DIR,
    '--parent-f-alpha', '0.15',
    '--output-json', f'artifacts/{RUN_ID}_robust.json',
    '--output-markdown', f'artifacts/{RUN_ID}_robust.md',
    enabled=RUN_HEAVY, label='dependence-aware robust evaluation',
)

## v17 pressure residual 학습 경로

초기 v17 recipe의 학습 명령을 DRY_RUN으로 확인합니다.

In [ ]:
FINAL_ARTIFACT_DIR = f'{RUN_ID}_residual_final'
run_module(
    'src.train_v16_residual',
    '--output-dir', f'artifacts/{FINAL_ARTIFACT_DIR}',
    '--group', 'pitcher_batter_hand_pressure',
    '--decay', '1',
    '--alpha', '3200',
    '--correction-weight', '1.5',
    '--candidate-name', RUN_ID,
    enabled=RUN_HEAVY, label='fit final pressure EB spec',
)

## v17 구조 패키징·검증 경로

초기 v14 부모에 residual spec 한 개를 추가하는 과거 패키징 계약입니다. 최종 v345 패키징에는 사용하지 않습니다.

In [ ]:
REBUILD_ZIP = f'submit_{RUN_ID}_rebuild.zip'
run_module(
    'src.package_v16_residual',
    '--parent', 'submit_v14.zip',
    '--output', REBUILD_ZIP,
    '--artifact-dir', FINAL_ARTIFACT_DIR,
    '--expected-parent-sha256', V14_SHA256,
    enabled=RUN_PACKAGING, label='package candidate',
)
run_module(
    'src.validate_v16_residual',
    '--candidate', REBUILD_ZIP,
    '--parent', 'submit_v14.zip',
    '--report', f'artifacts/{RUN_ID}_validation.md',
    enabled=RUN_PACKAGING, label='validate candidate package',
)

## 과거 워크벤치 종료 확인

이 노트북이 호출하는 v17 모듈은 역사적 회귀 검증 대상입니다. 새 산출물 생성이나 최종 v345 재현에는 루트 실행 노트북과 `03_REPRODUCE.md`를 사용합니다.